# DHBO-v2: run the whole study in Google Colab

Reproduces the results of the paper with the code in this repository.

1. Optional: Runtime → Change runtime type → GPU.
2. Set `FAST = True` in step 2 for a quick end-to-end check (about 30 minutes), or keep `False` for the full study (several hours).
3. Run all cells in order. Every step saves its results, so after a disconnection you can continue from the next step.

## 1. Code, libraries and data

In [ ]:
!git clone -q https://github.com/mohammedalnaggar2023-EG/DHBO-v2.git
%cd DHBO-v2
!pip install -q -r requirements.txt -r requirements_extra.txt
!mkdir -p data
!wget -q -O data/confirmed_global.csv https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv
!ls -la data

## 2. Settings

In [ ]:
FAST = False   # True = quick check, False = full study
F = '--fast' if FAST else ''
HELD = 'France Germany Japan "South Korea"'
import os; os.environ['PYTHONWARNINGS'] = 'ignore'

## 3. Development countries (India, Brazil, Russia, USA)

In [ ]:
!python run_experiments.py {F} --data data/confirmed_global.csv --out run_v1
!python run_v2.py {F} --data data/confirmed_global.csv --v1 run_v1 --out run_v2
!python run_v2.py {F} --data data/confirmed_global.csv --v1 run_v1 --corrector cnn --out run_cnn
!python analyze_v2.py 'run_v2/runs/*.json'

## 4. Held-out countries (same code and settings)

In [ ]:
!python run_experiments.py {F} --data data/confirmed_global.csv --countries {HELD} --out run_val_v1
!python run_v2.py {F} --data data/confirmed_global.csv --countries {HELD} --v1 run_val_v1 --out run_val_tcn
!python run_v2.py {F} --data data/confirmed_global.csv --countries {HELD} --v1 run_val_v1 --corrector cnn --out run_val_cnn

## 5. Three-design comparison

In [ ]:
!python compare_designs.py run_v2 run_cnn India,Brazil,Russia,USA run_v1 | tee compare_study.txt
!python compare_designs.py run_val_tcn run_val_cnn "France,Germany,Japan,South Korea" run_val_v1 | tee compare_unseen.txt

## 6. Robustness checks, N-HiTS and the earlier test period

These read the published result files in `results/`, so they can also be run without steps 3–5.

In [ ]:
!python robustness.py stat
!python robustness.py seeds
!python robustness.py arimalstm
!python nhits.py held
!bash run_period.sh

## 7. Figures

In [ ]:
!python make_figures.py
from IPython.display import Image, display
for f in ['Fig1', 'Fig2', 'Fig3', 'Fig4', 'Fig5', 'FigS1']:
    display(Image(f'figures/{f}.png', width=700))

## 8. Download everything

In [ ]:
!zip -qr dhbo_outputs.zip run_* compare_*.txt out_* nhits.json per_* figures
from google.colab import files
files.download('dhbo_outputs.zip')